In [1]:
from datetime import date
from pathlib import Path

import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

# Fecha final del rango de Analytics fijada en la extracción (checkpoint: 2026-10-20)
ANALYTICS_END = date(2026, 10, 20)

df = pd.read_csv(CSV_DIR / "video_performance.csv")
# utc=True convierte el texto ISO 8601 a fechas con zona horaria (UTC)
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)

# --- 1.1 Unicidad: una fila por vídeo --------------------------------------
n_rows = len(df)
n_unique = df["video_id"].nunique()
n_duplicated = int(df["video_id"].duplicated().sum())

print("== 1.1 Unicidad ==")
print(f"Filas: {n_rows} | video_id distintos: {n_unique} | duplicados: {n_duplicated}")
print("OK" if n_rows == n_unique == 375 and n_duplicated == 0 else "REVISAR")

# --- 1.2 Vídeos sin datos de Analytics (views nulo) ------------------------
sin_analytics = df[df["views"].isna()].sort_values("published_at", ascending=False)

print("\n== 1.2 Vídeos sin Analytics ==")
print(f"Cantidad: {len(sin_analytics)} (esperado: 4)")
print(sin_analytics[["video_id", "title", "published_at", "view_count"]].to_string(index=False))

# --- 1.3 ¿Son los más recientes y posteriores al fin del rango? ------------
n_missing = len(sin_analytics)
mas_recientes = df.nlargest(n_missing, "published_at")

son_los_mas_recientes = set(sin_analytics["video_id"]) == set(mas_recientes["video_id"])
todos_posteriores = bool(
    (sin_analytics["published_at"].dt.date > ANALYTICS_END).all()
)

print("\n== 1.3 Coherencia ==")
print(f"¿Son exactamente los {n_missing} vídeos más recientes?: {son_los_mas_recientes}")
print(f"¿Todos publicados después del {ANALYTICS_END}?: {todos_posteriores}")

== 1.1 Unicidad ==
Filas: 375 | video_id distintos: 375 | duplicados: 0
OK

== 1.2 Vídeos sin Analytics ==
Cantidad: 4 (esperado: 4)
   video_id                                                                       title              published_at  view_count
ebiJjFAypAI   Cuando tu Casa es un MUSEO de camisetas del BETIS con casi ¡150 unidades! 2026-10-23 10:12:35+00:00         197
22DOOQMCXE4              Madrid y Barça: los dos grandes perseguidos del fútbol español 2026-10-22 12:14:23+00:00           3
xIRZHwfq96A El Betis de Champions vale ORO: mucho más que tres puntos ante el Feyenoord 2026-10-22 09:00:12+00:00        9649
HiBmLGIwdIc                                        Cuando tu casa es un museo del BETIS 2026-10-21 17:36:00+00:00           4

== 1.3 Coherencia ==
¿Son exactamente los 4 vídeos más recientes?: True
¿Todos publicados después del 2026-10-20?: True
